 Os arquivos e seus respectivos:

 1. *custos* é o *input_losgistic_costs*, representa a matriz de custos de frete (combustível, pedágio e mão de
obra) para cada arco origem-destino. 
 2. *demanda_prazos* é o *input_production_need*, representa a necessidade volumétrica por SKU e respectivos prazos
finais (deadlines).
 3. *produção* é o *input_production_rate*, representa as taxas nominais de produção por linha e unidade (latas/
hora). 

In [1]:
import pandas as pd

## Informações relacionadas a *custos*

In [2]:
custos = pd.read_csv('custos.csv')
custos

,origem,destino,distancia_km,custo_combustivel,custo_pedagio,custo_mao_de_obra,custo_total_frete
0,Jacarei,Jacarei,0.00,0.00,0.00,0.00,0.00
1,Jacarei,Extrema,79.00,276.48,94.79,165.89,537.17
2,Jacarei,Tres Rios,390.57,1367.01,468.69,820.20,2655.90
3,Jacarei,Alagoinhas,1840.35,6441.21,2208.42,3864.73,12514.36
4,Jacarei,Frutal,592.57,2074.01,711.09,1244.41,4029.51
...,...,...,...,...,...,...,...
95,Cabo de Santo Agostinho,Pouso Alegre,2421.82,8476.37,2906.19,5085.82,16468.38
96,Cabo de Santo Agostinho,Gama,2064.82,7226.87,2477.78,4336.12,14040.78
97,Cabo de Santo Agostinho,Viamao,3668.73,12840.55,4402.47,7704.33,24947.36
98,Cabo de Santo Agostinho,Itupeva,2608.80,9130.81,3130.56,5478.49,17739.86


## Informações relacionadas a *demanda_prazos*

In [27]:
demanda_prazos = pd.read_csv('demanda_prazos.csv')
demanda_prazos

,id,scenario_id,production_line,plant,customer,order_number,product_id,deadline,prod_need
0,1,200,FRU_L3,Frutal,NaN,NaN,H2OH_Limao,2026-12-12 06:00:00.000,"1589,69"
1,2,200,EXT_L3,Extrema,NaN,NaN,Brahma_Lata,2026-12-02 06:00:00.000,"1170,48"
2,3,200,POU_L1,Pouso Alegre,NaN,NaN,Schweppes_Citrus,2026-12-15 06:00:00.000,"625,66"
3,4,200,EXT_L1,Extrema,NaN,NaN,Brahma_Lata,2026-12-12 06:00:00.000,"1106,29"
4,5,200,VIA_L2,Viamao,NaN,NaN,Brahma_Lata,2026-12-10 06:00:00.000,"1788,79"
...,...,...,...,...,...,...,...,...,...
2995,2996,200,JAC_L2,Jacarei,NaN,NaN,Del_Valle_Pessego,2026-12-06 06:00:00.000,"704,26"
2996,2997,200,POU_L2,Pouso Alegre,NaN,NaN,Pepsi,2026-12-11 06:00:00.000,"494,80"
2997,2998,200,ITU_L3,Itupeva,NaN,NaN,Del_Valle_Uva,2026-12-05 06:00:00.000,"0,00"
2998,2999,200,EXT_L3,Extrema,NaN,NaN,Antarctica_Lata,2026-12-15 06:00:00.000,"1156,93"


Em resumo, cada linha diz: "Na fábrica X, linha Y, o produto Z precisa ser produzido em quantidade N até a data D." (agora oq esta definido abaixo)

## Informações relacionadas a *produção*

In [ ]:
produção = pd.read_csv('produção.csv')
produção

Em resumo, cada linha do arquivo diz: "Na fábrica X, linha Y, no dia Z, a taxa de produção foi de N unidades do produto 9.1 Oz Sleek." O arquivo cobre 15 dias de dezembro de 2026, com 3 linhas por planta e 6 plantas no total (450 registros).

## 1. Tratamento da Demanda e Extração de Conjuntos

**Limpeza Obrigatória do Padrão Numérico**

• O que deve ser feito: A base apresenta os volumes de demanda no padrão de grafia brasileiro,
utilizando vírgulas como separadores decimais (ex: 2611, 16).

• Vocês devem tratar textualmente essa coluna antes de alimentar o modelo, substituindo a
vírgula pelo ponto decimal e convertendo o dado para o float

In [28]:
demanda_prazos['prod_need'] = demanda_prazos['prod_need'].astype(str).str.replace(',', '.').astype(float)
demanda_prazos

,id,scenario_id,production_line,plant,customer,order_number,product_id,deadline,prod_need
0,1,200,FRU_L3,Frutal,NaN,NaN,H2OH_Limao,2026-12-12 06:00:00.000,1589.69
1,2,200,EXT_L3,Extrema,NaN,NaN,Brahma_Lata,2026-12-02 06:00:00.000,1170.48
2,3,200,POU_L1,Pouso Alegre,NaN,NaN,Schweppes_Citrus,2026-12-15 06:00:00.000,625.66
3,4,200,EXT_L1,Extrema,NaN,NaN,Brahma_Lata,2026-12-12 06:00:00.000,1106.29
4,5,200,VIA_L2,Viamao,NaN,NaN,Brahma_Lata,2026-12-10 06:00:00.000,1788.79
...,...,...,...,...,...,...,...,...,...
2995,2996,200,JAC_L2,Jacarei,NaN,NaN,Del_Valle_Pessego,2026-12-06 06:00:00.000,704.26
2996,2997,200,POU_L2,Pouso Alegre,NaN,NaN,Pepsi,2026-12-11 06:00:00.000,494.80
2997,2998,200,ITU_L3,Itupeva,NaN,NaN,Del_Valle_Uva,2026-12-05 06:00:00.000,0.00
2998,2999,200,EXT_L3,Extrema,NaN,NaN,Antarctica_Lata,2026-12-15 06:00:00.000,1156.93


**Identificação dos Conjuntos**

A partir dos dados limpos, o grupo deve extrair as listas de elementos únicos que servirão como
os índices das variáveis e restrições matemáticas:

• **Conjunto *I* (SKUs)**: Isolar cada produto final específico exigido pelo mercado.

• **Conjunto *L* (Linhas)**: Mapear todas as linhas de produção industriais ativas.

• **Conjunto *P* (Plantas)**: Listar todas as fábricas geográficas envolvidas na malha.

• **Conjunto *T* (Horizonte de Tempo)**: Estruturar a linha do tempo do planejamento.

In [29]:
conjunto_I = demanda_prazos['product_id'].unique()
conjunto_L = demanda_prazos['production_line'].unique()
conjunto_P = demanda_prazos['plant'].unique()
conjunto_T = demanda_prazos['deadline'].unique()

In [30]:
conjunto_I, conjunto_L

(<StringArray>
 [             'H2OH_Limao',             'Brahma_Lata',
         'Schweppes_Citrus',          'Sukita_Laranja',
            'Del_Valle_Uva',             'Spaten_Lata',
            'Heineken_Lata',       'Del_Valle_Laranja',
              'Amstel_Lata',                   'Pepsi',
                'Coca_Cola', 'Stella_Artois_Pure_Gold',
            'Itaipava_Lata',               'Fanta_Uva',
        'Del_Valle_Pessego',             'Sprite_Zero',
                'Skol_Lata',      'Guarana_Antarctica',
         'Eisenbahn_Pilsen',                    'Kuat',
              'Pepsi_Black',                  'Sprite',
           'Coca_Cola_Zero',                'Itubaina',
  'Guarana_Antarctica_Zero',          'Budweiser_Lata',
            'Fanta_Laranja',         'Antarctica_Lata',
             'Maguary_Caju',        'Maguary_Maracuja']
 Length: 30, dtype: str,
 <StringArray>
 ['FRU_L3', 'EXT_L3', 'POU_L1', 'EXT_L1', 'VIA_L2', 'ALA_L3', 'EXT_L2',
  'JAC_L2', 'POU_L2', 'ITU_L1', '

In [31]:
conjunto_P, conjunto_T

(<StringArray>
 [                 'Frutal',                 'Extrema',
             'Pouso Alegre',                  'Viamao',
               'Alagoinhas',                 'Jacarei',
                  'Itupeva', 'Cabo de Santo Agostinho',
                     'Gama',               'Tres Rios']
 Length: 10, dtype: str,
 <StringArray>
 ['2026-12-12 06:00:00.000', '2026-12-02 06:00:00.000',
  '2026-12-15 06:00:00.000', '2026-12-10 06:00:00.000',
  '2026-12-04 06:00:00.000', '2026-12-14 06:00:00.000',
  '2026-12-08 06:00:00.000', '2026-12-05 06:00:00.000',
  '2026-12-06 06:00:00.000', '2026-12-07 06:00:00.000',
  '2026-12-13 06:00:00.000', '2026-12-01 06:00:00.000',
  '2026-12-03 06:00:00.000', '2026-12-09 06:00:00.000',
  '2026-12-11 06:00:00.000']
 Length: 15, dtype: str)

Os arquivos originais contêm carimbos de data/hora civis reais (ex: 2026-12-01
06:00:00). É comum que modelos de programação linear contínua dependem de índices
discretos ordenados de forma sequencial.

**O que fazer:** Filtrem todas as datas únicas de entrega, organizem-nas em ordem
estritamente cronológica e associem cada uma a um rótulo indexado sequencial
*(t01, t02, . . . , t15)*. Construam um mapeamento para garantir que, ao ler uma data real na
tabela, o algoritmo saiba exatamente a qual período *(t)* ela pertence.

In [32]:
datas_unicas = sorted(demanda_prazos['deadline'].unique())

rotulos = [f't{str(i+1).zfill(2)}' for i in range(len(datas_unicas))]

mapeamento_T = dict(zip(datas_unicas, rotulos))

print(mapeamento_T)

#isso cria uma nova coluna 'periodo_t' no DataFrame 'demanda_prazos', onde cada valor é o resultado da aplicação do mapeamento definido em 'mapeamento_T' à coluna 'deadline'. Ou seja, para cada valor na coluna 'deadline', ele será substituído pelo rótulo correspondente definido no dicionário 'mapeamento_T'.    
demanda_prazos['periodo_t'] = demanda_prazos['deadline'].map(mapeamento_T)

{'2026-12-01 06:00:00.000': 't01', '2026-12-02 06:00:00.000': 't02', '2026-12-03 06:00:00.000': 't03', '2026-12-04 06:00:00.000': 't04', '2026-12-05 06:00:00.000': 't05', '2026-12-06 06:00:00.000': 't06', '2026-12-07 06:00:00.000': 't07', '2026-12-08 06:00:00.000': 't08', '2026-12-09 06:00:00.000': 't09', '2026-12-10 06:00:00.000': 't10', '2026-12-11 06:00:00.000': 't11', '2026-12-12 06:00:00.000': 't12', '2026-12-13 06:00:00.000': 't13', '2026-12-14 06:00:00.000': 't14', '2026-12-15 06:00:00.000': 't15'}


## 2. Estruturação dos Parâmetros e Hierarquias

**Dicionário de Demanda:**

Cruzem a tabela de necessidades para gerar uma estrutura tridimensional. Ao prover um
Produto específico *(i)*, uma Linha de referência *(l)* e um Período *(t)*, a estrutura deve retornar
de forma imediata o volume exato de latas demandado.

In [ ]:
demanda = {}

for _, row in demanda_prazos.iterrows():
    i = row['product_id']
    l = row['production_line']
    t = mapeamento_T[row['deadline']]
    v = row['prod_need']
    
    demanda[(i, l, t)] = v

# O que é isso? é um dicionario 3d  A chave é uma tupla (i, l, t) onde:
# i é o id do produto, l é a linha de produção, t é o período (rótulo do prazo)
# Ao inserir os três elementos na tupla, o valor associado a essa chave é a quantidade necessária do produto i na linha de produção l para o período t.

#Exemplo: Pegar a primeira chave existente
chave = list(demanda.keys())[0]
print(chave)
print(demanda[chave])


('H2OH_Limao', 'FRU_L3', 't12')
1589.69


**Mapeamento da Hierarquia Física e Geográfica:**

As linhas de produção estão fisicamente contidas dentro das fábricas. Construam uma estrutura
de mapeamento que receba uma Linha *(l)* e retorne a sua Planta *(p)* de origem (ex: ao consultar
JAC_L1, o sistema deve mapear para Jacarei).

In [48]:
linha_para_planta = dict(zip(demanda_prazos['production_line'], demanda_prazos['plant']))

#Exemplos:
print(linha_para_planta['JAC_L1'])
print(linha_para_planta['EXT_L2'])

Jacarei
Extrema


## 3. Capacidades de Manufatura e Logística

### Matriz de Velocidade da Linha:

A tabela de taxas de produção (input_production_rate_artificial.csv) expressa a produtividade
física em latas por hora.

**Filtro de Consistência:** Ao processar este arquivo, verifiquem se a data de referência
existe no horizonte de planejamento mapeado anteriormente. Datas incompatíveis devem ser
ignoradas para evitar quebras de consistência no código.

In [50]:
datas_validas = set(mapeamento_T.keys())

taxas_validas = produção[produção['ref_date'].isin(datas_validas)]

print(f"Linhas antes do filtro: {len(produção)}")
print(f"Linhas após o filtro:   {len(taxas_validas)}")

Linhas antes do filtro: 450
Linhas após o filtro:   450


**Definição da Abordagem de Capacidade:** Na formulação das restrições de capacidade,
cabe ao grupo escolher qual perspectiva conceitual adotar no modelo, sendo possível seguir
pelo caminho de mensurar a ocupação física das linhas em tempo (garantindo que o uso não
ultrapasse o limite do período) ou modelar o teto físico diretamente em quantidade/volume.
É fundamental que a escolha da abordagem seja acompanhada de uma adequação imediata
na definição das variáveis de decisão e dos parâmetros do modelo. Lembrem-se de que, sob
o regime de turnos ininterruptos, a fábrica conta com a totalidade das horas do período
disponível (ex: 12 horas por turno), cabendo ao grupo garantir a completa coerência das
unidades de medida em toda a modelagem.

Essa parte é uma **decisão de modelagem** que seu grupo precisa tomar. Deixa eu explicar as duas abordagens:

---

**Opção 1 — Restrição em Tempo (horas)**

A variável de decisão seria **quanto tempo** alocar cada linha para cada produto:
- Variável: `x[i,l,t]` = horas dedicadas ao produto i, na linha l, no período t
- Restrição: a soma das horas usadas não pode ultrapassar as horas disponíveis (ex: 12h por turno)
- Para isso, você precisaria saber a **taxa em latas/hora** para converter produção em tempo

---

**Opção 2 — Restrição em Volume (latas/unidades)**

A variável de decisão seria **quanto produzir** de cada produto:
- Variável: `x[i,l,t]` = volume produzido do produto i, na linha l, no período t
- Restrição: a soma dos volumes não pode ultrapassar a capacidade máxima da linha no período
- A `rate` já seria diretamente o teto de volume

---

A diferença prática é:

| | Tempo | Volume |
|---|---|---|
| Variável | horas alocadas | unidades produzidas |
| Restrição | `Σ horas ≤ 12h` | `Σ volume ≤ rate` |
| Precisa converter? | Sim (taxa → tempo) | Não |


### Matriz de Custos de Transferência:

Recomendo que a base de custos logísticos (input_logistic_costs_artificial.csv) seja
modelada como uma matriz de adjacência entre localidades.

**Fluxo Interno:** Quando a origem for idêntica ao destino, fixem o custo em zero (0.00),
caracterizando que o produto atende ao consumo ou estocagem da própria planta produtora.

In [ ]:
custo_logistico = {}

for _, row in custos.iterrows():
    origem  = row['origem']
    destino = row['destino']
    custo   = row['custo_total_frete']
    
    # Fluxo interno: origem == destino -> custo zero (já está assim no arquivo, mas garantimos)
    if origem == destino:
        custo = 0.0
    
    custo_logistico[(origem, destino)] = custo

#Exemplos:
print(custo_logistico[('Jacarei', 'Extrema')])
print(custo_logistico[('Jacarei', 'Jacarei')])
print(custo_logistico[('Viamao', 'Cabo de Santo Agostinho')])

537.17
0.0
24947.36


**Fluxo de Rede:** Para arcos entre localidades distintas, agreguem os componentes de combustível,
pedágios e mão de obra para definir o custo total de frete do arco de transferência. 

*Nota:* isso já é dado na tabela *custos*, não sei para que calcular novamente.

In [57]:
custo_logistico = {}

for _, row in custos.iterrows():
    origem  = row['origem']
    destino = row['destino']
    
    if origem == destino:
        custo = 0.0
    else:
        custo = round(row['custo_combustivel'] + row['custo_pedagio'] + row['custo_mao_de_obra'], 2)
    
    custo_logistico[(origem, destino)] = custo

#Exemplo:
print(custo_logistico[('Jacarei', 'Extrema')])

537.16
